### 1. 영상 정보 확인

In [ ]:
import cv2

VIDEO = "../data/external/sample_15s.mp4"

cap = cv2.VideoCapture(VIDEO)
print("열림:", cap.isOpened())
print("해상도:", int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), "x", int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
print("FPS:", cap.get(cv2.CAP_PROP_FPS))
print("총 프레임:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
cap.release()

### 2. 첫 프레임 추론

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt

model = YOLO("yolo11n-pose.pt")

cap = cv2.VideoCapture(VIDEO)
ok, frame = cap.read()
cap.release()
print("프레임 읽기 성공:", ok, "/ shape:", frame.shape)

results = model(frame)
r = results[0]

plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()
           

In [ ]:
print(type(r))
print("원본 크기:", r.orig_shape)
print("걸린 시간(ms):", r.speed)
print("클래스:", r.names)
print("박스 수:", len(r.boxes))
print("박스 좌표(x1,y1,x2,y2):", r.boxes.xyxy)
print("박스 신뢰도:", r.boxes.conf)

In [ ]:
k = r.keypoints

print(k.xy.shape)     # (사람 수, 17, 2)  픽셀 좌표
print(k.conf.shape)   # (사람 수, 17)     관절별 신뢰도
print(k.xyn.shape)    # (사람 수, 17, 2)  0~1로 정규화된 좌표
print(k.data.shape)   # (사람 수, 17, 3)  x, y, conf를 합친 것

In [ ]:
import numpy as np

xy = r.keypoints.xy[0].cpu().numpy()
conf = r.keypoints.conf[0].cpu().numpy()

img = cv2.cvtColor(r.orig_img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(8, 8))
plt.imshow(img)
for i, (x, y) in enumerate(xy):
    if conf[i] > 0.3:
        plt.scatter(x, y, s=20, c="yellow")
        plt.text(x + 5, y - 5, str(i), color="cyan", fontsize=10)
plt.axis("off")
plt.show()

In [ ]:
names = {0:"코", 1:"왼눈", 2:"오른눈", 3:"왼귀", 4:"오른귀",
         5:"왼어깨", 6:"오른어깨", 7:"왼팔꿈치", 8:"오른팔꿈치",
         9:"왼손목", 10:"오른손목"}
for i, n in names.items():
    print(f"{i:2d} {n:6s} x={xy[i,0]:6.1f} y={xy[i,1]:6.1f} conf={conf[i]:.2f}")